# Defect Detection — Colab Training (Phase 2 scale-up)

This notebook re-runs the training that was done locally on a 4GB laptop GPU, but with the bigger models / longer schedules that only make sense on a real GPU (Colab's free T4 has 15GB VRAM; A100 if you have Pro).

Local results this notebook aims to beat (see `DEVELOPMENT_JOURNEY.md` for the full story):

| Model | Local (laptop, 4GB) | This notebook |
|---|---|---|
| PCB YOLO | yolov8n, 5 epochs — mAP50 0.978 | yolov8s, 100 epochs |
| Packaging YOLO | yolov8n, 5 epochs — mAP50 0.810 | yolov8s, 100 epochs |
| PatchCore (metal_nut) | resnet18, 256px — AUROC 0.9946 | wide_resnet50_2, 512px |

**How to use this notebook:**
1. `Runtime -> Change runtime type -> GPU` (T4 is fine).
2. Run the **Setup** cells.
3. Paste your own Roboflow API key in the **Secrets** cell (get one free at https://app.roboflow.com -> Settings -> Roboflow API). Never commit a real key to git — this cell is a placeholder, fill it in only inside your running Colab session.
4. Run **Part 1** (YOLO) and/or **Part 2** (PatchCore) — they're independent, run either or both.
5. Run the final **Download weights** cell and copy the files back into your local repo (paths are printed at the end).

## Setup

In [ ]:
# Confirm a GPU is attached (Runtime -> Change runtime type -> GPU if this errors)
!nvidia-smi

In [ ]:
import os

REPO_URL = "https://github.com/Anshumaan-25/Defect-Detection-Tech-M.git"

if not os.path.exists("Defect-Detection-Tech-M"):
    !git clone {REPO_URL}
%cd Defect-Detection-Tech-M

In [ ]:
# Colab already ships a CUDA-enabled torch, so this mainly installs the
# packages torch/torchvision won't be reinstalled since requirements.txt
# only pins a minimum version that Colab's build already satisfies.
!pip install -q -r requirements.txt

## Secrets — Roboflow API key

Only needed for **Part 1** (YOLO datasets come from Roboflow). Get a free key at
https://app.roboflow.com -> Settings -> Roboflow API -> Private API Key.

**Do not commit this notebook after pasting a real key into the cell below** —
keep the fill-in local to your Colab session.

In [ ]:
import os
os.environ["ROBOFLOW_API_KEY"] = "PASTE_YOUR_ROBOFLOW_API_KEY_HERE"

## Part 1 — Bigger, longer YOLO retrain (PCB + packaging)

Local training used `yolov8n` for 5 epochs because of the 4GB VRAM ceiling and
the GPU periodically dropping (Windows Optimus disabling it on low battery — see
`DEVELOPMENT_JOURNEY.md` §4 and §7). Colab removes both constraints, so this
uses `yolov8s` (bigger backbone) and 100 epochs.

Run names are kept **identical to the local runs** (`pcb_defects_yolo`,
`packaging_damage_yolo`) so bringing the weights home is a plain file swap —
just overwrite `models/yolo/<name>/weights/best.pt` locally, no code changes.

In [ ]:
!python -m src.dataset_loader --name pcb_defects
!python -m src.dataset_loader --name packaging_damage

In [ ]:
!python -m src.train_yolo \
    --dataset pcb_defects \
    --model yolov8s.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 32 \
    --workers 8 \
    --device 0 \
    --name pcb_defects_yolo

In [ ]:
!python -m src.train_yolo \
    --dataset packaging_damage \
    --model yolov8s.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 32 \
    --workers 8 \
    --device 0 \
    --name packaging_damage_yolo

## Part 2 — PatchCore scale-up (bigger backbone)

MVTec AD is **research-license gated** — it can't be auto-downloaded, on
Colab or anywhere else. You need your own copy of the archive(s) uploaded to
Google Drive first (register + download from
https://www.mvtec.com/company/research/datasets/mvtec-ad).

This section mounts Drive, copies the archive(s) in, and trains with
`wide_resnet50_2` (vs. `resnet18` locally) at 512px (vs. 256px locally) —
the bigger config the 4GB laptop GPU couldn't fit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pathlib
import shutil

# --- EDIT THIS: where you uploaded MVTec archives in your Drive ---
DRIVE_MVTEC_DIR = "/content/drive/MyDrive/mvtec_ad"

target_dir = pathlib.Path("data/raw/anomaly/mvtec_ad")
target_dir.mkdir(parents=True, exist_ok=True)

archive = pathlib.Path(DRIVE_MVTEC_DIR) / "metal_nut.tar.xz"
if archive.exists():
    shutil.copy(archive, target_dir / archive.name)
    print(f"Copied {archive}")
else:
    print(f"Not found: {archive} -- update DRIVE_MVTEC_DIR above to match your Drive layout")

In [ ]:
!python -m src.dataset_loader --name mvtec_ad

In [ ]:
!python -m src.train_anomaly \
    --category metal_nut \
    --backbone wide_resnet50_2 \
    --image-size 512 \
    --batch 32 \
    --device 0 \
    --name mvtec_ad_metal_nut_patchcore

### Part 2b (optional) — more MVTec categories

To train categories beyond `metal_nut` (Phase 2 roadmap item), upload their
archives (e.g. `bottle.tar.xz`, `cable.tar.xz`) into the same Drive folder,
list them below, and run this cell. Each category becomes its own run under
`models/anomaly/mvtec_ad_<category>_patchcore/`.

In [ ]:
# --- EDIT THIS: categories you've uploaded archives for ---
EXTRA_CATEGORIES = []  # e.g. ["bottle", "cable"]

for cat in EXTRA_CATEGORIES:
    archive = pathlib.Path(DRIVE_MVTEC_DIR) / f"{cat}.tar.xz"
    if archive.exists():
        shutil.copy(archive, target_dir / archive.name)
    else:
        print(f"Skipping {cat}: archive not found at {archive}")

if EXTRA_CATEGORIES:
    !python -m src.dataset_loader --name mvtec_ad
    for cat in EXTRA_CATEGORIES:
        !python -m src.train_anomaly --category {cat} --backbone wide_resnet50_2 --image-size 512 --batch 32 --device 0 --name mvtec_ad_{cat}_patchcore

## Download trained weights

Zips everything under `models/` and downloads it. Unzip locally and copy the
relevant folders into your repo's `models/yolo/` and `models/anomaly/` — same
names as local, so it's a drop-in replacement (no code changes needed).

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("trained_models", "zip", "models")
files.download("trained_models.zip")

## Bringing results home

1. Unzip `trained_models.zip`.
2. Copy `yolo/pcb_defects_yolo/weights/best.pt` -> local `models/yolo/pcb_defects_yolo/weights/best.pt` (overwrite).
3. Copy `yolo/packaging_damage_yolo/weights/best.pt` -> local `models/yolo/packaging_damage_yolo/weights/best.pt` (overwrite).
4. Copy `anomaly/mvtec_ad_metal_nut_patchcore/weights/torch/model.pt` -> local equivalent (overwrite).
5. Re-run `python app.py` locally — no code changes needed, since run names match.
6. Update `DEVELOPMENT_JOURNEY.md` with the new results (per the standing protocol at the top of that file).